<a href="https://colab.research.google.com/github/Mosizamani/aai_530_final_project_group_4/blob/master/ai_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ucimlrepo


In [13]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# Define parameters for windowing
WINDOW_SIZE = 256  # Number of time steps in each sequence (e.g., 4 seconds at 64Hz)
STEP_SIZE = 128     # Number of time steps to slide the window (e.g., 2 seconds overlap)

def create_windows(data, labels, window_size, step_size):
    X, y = [], []
    for i in range(0, len(data) - window_size, step_size):
        X.append(data[i:i + window_size])
        # For labels, we can take the mode, mean, or the last value in the window
        # For stress detection, often the most frequent label in the window is used
        # Or, if events are distinct, the label at the end of the window
        # Let's take the mode for simplicity for now, handling potential empty modes
        window_labels = labels[i:i+window_size]
        if len(window_labels) > 0:
            modes = pd.Series(window_labels).mode()
            if not modes.empty:
                y.append(modes.iloc[0]) # Take the first mode if multiple
            else:
                y.append(np.nan) # Handle case where no mode can be found (e.g., all nan in window)
        else:
            y.append(np.nan)

    # Filter out windows where label is NaN if they occurred
    valid_indices = ~np.isnan(y)
    X = np.array(X)[valid_indices]
    y = np.array(y)[valid_indices]

    return X, y

# Prepare data for all subjects
X_all, y_all = [], []
subject_scalers = {}

# Iterate through each subject's processed data
for subject_name, df_subject in processed_subject_data.items():
    print(f"\n--- Windowing and Scaling data for {subject_name} ---")

    # Separate features and labels
    features = df_subject.drop(columns=['label'])
    labels = df_subject['label'].values

    # Handle potential NaNs in features due to outer merge and fillna issues
    # For this specific dataset, we expect numerical data. Let's fill any remaining NaNs with 0.
    features = features.fillna(0)

    # Scale features for the current subject
    # It's important to fit the scaler only on the training data later to avoid data leakage
    # However, for now, we'll scale per subject based on their full data for consistency
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    subject_scalers[subject_name] = scaler # Store scaler if we need to inverse transform later

    # Create windows for the current subject
    X_subject, y_subject = create_windows(scaled_features, labels, WINDOW_SIZE, STEP_SIZE)

    if X_subject.size > 0:
        X_all.append(X_subject)
        y_all.append(y_subject)
        print(f"  Created {len(X_subject)} windows for {subject_name}")
    else:
        print(f"  No valid windows created for {subject_name}")

# Concatenate all subjects' data
if X_all and y_all:
    X_combined = np.vstack(X_all)
    y_combined = np.concatenate(y_all)
    print(f"\nCombined data from all subjects: X_combined shape {X_combined.shape}, y_combined shape {y_combined.shape}")

    # Split data into training and testing sets
    # We'll need to decide on a robust splitting strategy, e.g., subject-wise split or time-based split
    # For a simple start, let's do a random split on combined windows
    X_train, X_test, y_train, y_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined)

    print(f"\nTraining data shape: {X_train.shape}, {y_train.shape}")
    print(f"Testing data shape: {X_test.shape}, {y_test.shape}")

    # Display unique labels and their counts to check stratification
    unique_labels, counts = np.unique(y_train, return_counts=True)
    print(f"Training label distribution: {dict(zip(unique_labels, counts))}")
    unique_labels, counts = np.unique(y_test, return_counts=True)
    print(f"Testing label distribution: {dict(zip(unique_labels, counts))}")

else:
    print("No data available after processing subjects to create combined dataset.")



--- Windowing and Scaling data for S2 ---
  Created 45 windows for S2

--- Windowing and Scaling data for S3 ---
  Created 47 windows for S3

--- Windowing and Scaling data for S4 ---
  Created 47 windows for S4

--- Windowing and Scaling data for S5 ---
  Created 45 windows for S5

--- Windowing and Scaling data for S6 ---
  Created 51 windows for S6

--- Windowing and Scaling data for S7 ---
  Created 40 windows for S7

--- Windowing and Scaling data for S8 ---
  Created 41 windows for S8

--- Windowing and Scaling data for S9 ---
  Created 40 windows for S9

--- Windowing and Scaling data for S10 ---
  Created 42 windows for S10

--- Windowing and Scaling data for S11 ---
  Created 40 windows for S11

--- Windowing and Scaling data for S13 ---
  Created 42 windows for S13

--- Windowing and Scaling data for S14 ---
  Created 32 windows for S14

--- Windowing and Scaling data for S15 ---
  Created 40 windows for S15

--- Windowing and Scaling data for S16 ---
  Created 43 windows fo

In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

# Assuming y_train and y_test contain integer labels, but they might not be 0-indexed
# First, let's map the labels to a 0-indexed range
# Identify unique labels and map them
all_unique_labels = np.unique(y_combined)
label_mapping = {label: i for i, label in enumerate(all_unique_labels)}

y_train_mapped = np.array([label_mapping[label] for label in y_train])
y_test_mapped = np.array([label_mapping[label] for label in y_test])

# Convert integer labels to one-hot encoded vectors
num_classes = len(all_unique_labels)
y_train_one_hot = to_categorical(y_train_mapped, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test_mapped, num_classes=num_classes)

# Define the LSTM model
model = keras.Sequential([
    layers.LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax') # Output layer for multi-class classification
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display model summary
print("\n--- LSTM Model Summary ---")
model.summary()

# Train the model
print("\n--- Training LSTM Model ---")
history = model.fit(
    X_train,
    y_train_one_hot,
    epochs=20, # You might want to adjust the number of epochs
    batch_size=32,
    validation_split=0.2, # Use a portion of training data for validation
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

# Evaluate the model
print("\n--- Evaluating LSTM Model ---")
loss, accuracy = model.evaluate(X_test, y_test_one_hot, verbose=1)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

# Store metrics for potential future use or plotting
lstm_history = history
lstm_test_loss = loss
lstm_test_accuracy = accuracy



--- LSTM Model Summary ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 256, 64)        │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,378 (208.51 KB)

 Trainable params: 53,378 (208.51 KB)

 Non-trainable params: 0 (0.00 B)


--- Training LSTM Model ---
Epoch 1/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.6551 - loss: 0.6322 - val_accuracy: 0.9320 - val_loss: 0.3151
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9658 - loss: 0.1978 - val_accuracy: 0.9320 - val_loss: 0.2147
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9624 - loss: 0.1240 - val_accuracy: 0.9612 - val_loss: 0.0978
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9901 - loss: 0.0535 - val_accuracy: 0.9903 - val_loss: 0.0614
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9877 - loss: 0.0420 - val_accuracy: 1.0000 - val_loss: 0.0264
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9967 - loss: 0.0133 - val_accuracy: 0.9903 - val_loss: 0.0388
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9941 - loss: 0.0210 - val_accuracy: 1.0000 - val_loss: 0.0056
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9956 - loss: 0.01

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import resample

# Target frequency for resampling all sensor data
TARGET_FREQ = 32  # Hz

# Dictionary to store processed DataFrames for each subject
processed_subject_data = {}

# Original frequencies for sensors (these are hardcoded based on WESAD dataset info)
CHEST_ACC_ORIG_FREQ = 700  # Hz
CHEST_OTHER_ORIG_FREQ = 700 # ECG, EMG, EDA, Temp, Resp are also 700Hz
WRIST_ACC_ORIG_FREQ = 32   # Hz
WRIST_BVP_ORIG_FREQ = 64   # Hz
WRIST_EDA_ORIG_FREQ = 4    # Hz
WRIST_TEMP_ORIG_FREQ = 4   # Hz
LABEL_ORIG_FREQ = 700 # Labels are recorded at 700 Hz as per WESAD dataset


def resample_data(df, original_freq, target_freq, sensor_type=None):
    if df.empty:
        return df.copy()

    original_len = len(df)
    if original_len == 0:
        return df.copy()

    # Convert original_len to float for calculation to avoid integer division issues
    resample_len = int(original_len * float(target_freq) / original_freq)

    # Ensure resample_len is at least 1 if original_len > 0
    if resample_len == 0 and original_len > 0:
        resample_len = 1

    if resample_len == original_len:
        # No resampling needed if frequencies are the same
        resampled_df = df.copy()
    elif original_len > 1: # Resample only if there's enough data to resample
        resampled_data = {}
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                resampled_data[col] = resample(df[col].values, resample_len)
            else:
                # Handle non-numeric columns if necessary, for now, we'll skip or use a simple strategy
                # For WESAD, all sensor data is numeric
                pass
        resampled_df = pd.DataFrame(resampled_data, columns=df.columns)
    else:
        # If only one data point, cannot resample meaningfully, just return it
        resampled_df = df.copy()

    # Create a new TimedeltaIndex based on the target frequency
    resampled_df.index = pd.to_timedelta(np.arange(len(resampled_df)) / float(target_freq), unit='s')
    return resampled_df


print(f"Processing data for subjects to {TARGET_FREQ}Hz...")

# Iterate through each subject that has labels (indicating they were loaded successfully)
for subject_name in all_labels_data.keys():
    print(f"\n--- Processing {subject_name} ---")

    subject_chest_dataframes = {
        k.split(f'{subject_name}_')[1]: v
        for k, v in all_chest_dataframes.items()
        if k.startswith(f'{subject_name}_')
    }
    subject_wrist_dataframes = {
        k.split(f'{subject_name}_')[1]: v
        for k, v in all_wrist_dataframes.items()
        if k.startswith(f'{subject_name}_')
    }

    # Get questionnaire data for event timings (using the fallback mechanism)
    current_quest_df = all_quest_dataframes.get(subject_name)
    if current_quest_df is None:
        print(f"Warning: Questionnaire data not found for {subject_name}. Skipping event timing extraction.")
        # If no quest data, we cannot extract timings, so skip this subject or handle differently
        continue

    # --- 1. Extract Event Timings ---
    # This part is largely copied from previous logic in `pDmEu1G6BKNg` but localized
    order_str = current_quest_df.iloc[0, 0]
    start_str = current_quest_df.iloc[1, 0]
    end_str = current_quest_df.iloc[2, 0]

    def clean_split_list(s):
        parts = s.replace('#', '').split(';')
        return [p.strip() for p in parts if p.strip()]

    cleaned_order = clean_split_list(order_str)
    cleaned_start = clean_split_list(start_str)
    cleaned_end = clean_split_list(end_str)

    event_names = cleaned_order[1:]

    start_times_dict = {}
    for i, event in enumerate(event_names):
        if (i + 1) < len(cleaned_start):
            try:
                start_times_dict[event] = float(cleaned_start[i + 1])
            except ValueError:
                start_times_dict[event] = None

    end_times_dict = {}
    for i, event in enumerate(event_names):
        if (i + 1) < len(cleaned_end):
            try:
                end_times_dict[event] = float(cleaned_end[i + 1])
            except ValueError:
                end_times_dict[event] = None

    df_event_timings_subject = pd.DataFrame({
        'Event': event_names,
        'Start_Time': [start_times_dict.get(e) for e in event_names],
        'End_Time': [end_times_dict.get(e) for e in event_names]
    })
    # Convert to datetime for easier comparison and alignment
    df_event_timings_subject['Start_Time'] = pd.to_timedelta(df_event_timings_subject['Start_Time'], unit='s')
    df_event_timings_subject['End_Time'] = pd.to_timedelta(df_event_timings_subject['End_Time'], unit='s')

    # --- 2. Resample Chest Data ---
    resampled_chest_data = {}
    for sensor, df in subject_chest_dataframes.items():
        original_freq = CHEST_ACC_ORIG_FREQ if sensor == 'ACC' else CHEST_OTHER_ORIG_FREQ
        resampled_chest_data[sensor] = resample_data(df, original_freq, TARGET_FREQ)
        print(f"  Resampled chest {sensor} from {original_freq}Hz to {TARGET_FREQ}Hz. New shape: {resampled_chest_data[sensor].shape}")

    # --- 3. Resample Wrist Data ---
    resampled_wrist_data = {}
    wrist_orig_freq_map = {'ACC': WRIST_ACC_ORIG_FREQ, 'BVP': WRIST_BVP_ORIG_FREQ, 'EDA': WRIST_EDA_ORIG_FREQ, 'TEMP': WRIST_TEMP_ORIG_FREQ}
    for sensor, df in subject_wrist_dataframes.items():
        original_freq = wrist_orig_freq_map.get(sensor, TARGET_FREQ) # Default to TARGET_FREQ if not found
        resampled_wrist_data[sensor] = resample_data(df, original_freq, TARGET_FREQ)
        print(f"  Resampled wrist {sensor} from {original_freq}Hz to {TARGET_FREQ}Hz. New shape: {resampled_wrist_data[sensor].shape}")

    # --- 4. Process and Align Labels ---
    labels = all_labels_data[subject_name]
    # Labels are originally at 700Hz, we need to resample them to TARGET_FREQ

    # Create a DataFrame for labels to use the resample_data function
    df_labels_orig = pd.DataFrame(labels, columns=['label'])
    df_labels_resampled = resample_data(df_labels_orig, LABEL_ORIG_FREQ, TARGET_FREQ)
    print(f"  Resampled labels from {LABEL_ORIG_FREQ}Hz to {TARGET_FREQ}Hz. New shape: {df_labels_resampled.shape}")

    # --- 5. Consolidate and Align all data for the subject ---
    # Merge all resampled sensor dataframes and labels
    df_subject_merged = pd.DataFrame(index=pd.to_timedelta(np.arange(len(df_labels_resampled)) / float(TARGET_FREQ), unit='s'))

    # Add chest data
    for sensor, df in resampled_chest_data.items():
        df_subject_merged = df_subject_merged.merge(df, how='outer', left_index=True, right_index=True, suffixes=('', f'_{sensor}_chest'))

    # Add wrist data
    for sensor, df in resampled_wrist_data.items():
        df_subject_merged = df_subject_merged.merge(df, how='outer', left_index=True, right_index=True, suffixes=('', f'_{sensor}_wrist'))

    # Add labels, ensuring they are named 'label'
    df_subject_merged = df_subject_merged.merge(df_labels_resampled, how='outer', left_index=True, right_index=True, suffixes=('', '_label'))

    # Fill any missing values after outer merge, often due to slight length differences during resampling
    df_subject_merged = df_subject_merged.ffill().bfill() # Forward fill then backward fill

    # --- Clip data to event timings ---
    # Use the 'Base' event as the primary timing reference for the overall segment
    base_start_time = df_event_timings_subject[df_event_timings_subject['Event'] == 'Base']['Start_Time'].iloc[0]
    base_end_time = df_event_timings_subject[df_event_timings_subject['Event'] == 'Medi 2']['End_Time'].iloc[0] # Assuming Medi 2 is the last relevant event

    df_subject_final = df_subject_merged[
        (df_subject_merged.index >= base_start_time) &
        (df_subject_merged.index <= base_end_time)
    ].copy()

    processed_subject_data[subject_name] = df_subject_final
    print(f"  Final processed DataFrame for {subject_name} shape: {df_subject_final.shape}")
    print(f"  Columns for {subject_name}: {df_subject_final.columns.tolist()}")

print("\n--- All subjects processed. ---")
print("Keys in processed_subject_data:", processed_subject_data.keys())

# Display head of one processed subject's data for verification
if processed_subject_data:
    first_subject = list(processed_subject_data.keys())[0]
    print(f"\nHead of {first_subject}'s processed data:")
    print(processed_subject_data[first_subject].head())
    print(f"Info of {first_subject}'s processed data:")
    processed_subject_data[first_subject].info()


In [12]:
import pandas as pd
import numpy as np
from scipy.signal import resample

# Target frequency for resampling all sensor data
TARGET_FREQ = 64  # Hz

# Dictionary to store processed DataFrames for each subject
processed_subject_data = {}

# Original frequencies for sensors (these are hardcoded based on WESAD dataset info)
CHEST_ACC_ORIG_FREQ = 700  # Hz
CHEST_OTHER_ORIG_FREQ = 700 # ECG, EMG, EDA, Temp, Resp are also 700Hz
WRIST_ACC_ORIG_FREQ = 32   # Hz
WRIST_BVP_ORIG_FREQ = 64   # Hz
WRIST_EDA_ORIG_FREQ = 4    # Hz
WRIST_TEMP_ORIG_FREQ = 4   # Hz
LABEL_ORIG_FREQ = 700 # Labels are recorded at 700 Hz as per WESAD dataset


def resample_data(df, original_freq, target_freq, sensor_type=None):
    if df.empty:
        return df.copy()

    original_len = len(df)
    if original_len == 0:
        return df.copy()

    # Convert original_len to float for calculation to avoid integer division issues
    resample_len = int(original_len * float(target_freq) / original_freq)

    # Ensure resample_len is at least 1 if original_len > 0
    if resample_len == 0 and original_len > 0:
        resample_len = 1

    if resample_len == original_len:
        # No resampling needed if frequencies are the same
        resampled_df = df.copy()
    elif original_len > 1: # Resample only if there's enough data to resample
        resampled_data = {}
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                resampled_data[col] = resample(df[col].values, resample_len)
            else:
                # Handle non-numeric columns if necessary, for now, we'll skip or use a simple strategy
                # For WESAD, all sensor data is numeric
                pass
        resampled_df = pd.DataFrame(resampled_data, columns=df.columns)
    else:
        # If only one data point, cannot resample meaningfully, just return it
        resampled_df = df.copy()

    # Create a new TimedeltaIndex based on the target frequency
    resampled_df.index = pd.to_timedelta(np.arange(len(resampled_df)) / float(target_freq), unit='s')
    return resampled_df


print(f"Processing data for subjects to {TARGET_FREQ}Hz...")

# Iterate through each subject that has labels (indicating they were loaded successfully)
for subject_name in all_labels_data.keys():
    print(f"\n--- Processing {subject_name} ---")

    subject_chest_dataframes = {
        k.split(f'{subject_name}_')[1]: v
        for k, v in all_chest_dataframes.items()
        if k.startswith(f'{subject_name}_')
    }
    subject_wrist_dataframes = {
        k.split(f'{subject_name}_')[1]: v
        for k, v in all_wrist_dataframes.items()
        if k.startswith(f'{subject_name}_')
    }

    # Get questionnaire data for event timings (using the fallback mechanism)
    current_quest_df = all_quest_dataframes.get(subject_name)
    if current_quest_df is None:
        print(f"Warning: Questionnaire data not found for {subject_name}. Skipping event timing extraction.")
        # If no quest data, we cannot extract timings, so skip this subject or handle differently
        continue

    # --- 1. Extract Event Timings ---
    # This part is largely copied from previous logic in `pDmEu1G6BKNg` but localized
    order_str = current_quest_df.iloc[0, 0]
    start_str = current_quest_df.iloc[1, 0]
    end_str = current_quest_df.iloc[2, 0]

    def clean_split_list(s):
        parts = s.replace('#', '').split(';')
        return [p.strip() for p in parts if p.strip()]

    cleaned_order = clean_split_list(order_str)
    cleaned_start = clean_split_list(start_str)
    cleaned_end = clean_split_list(end_str)

    event_names = cleaned_order[1:]

    start_times_dict = {}
    for i, event in enumerate(event_names):
        if (i + 1) < len(cleaned_start):
            try:
                start_times_dict[event] = float(cleaned_start[i + 1])
            except ValueError:
                start_times_dict[event] = None

    end_times_dict = {}
    for i, event in enumerate(event_names):
        if (i + 1) < len(cleaned_end):
            try:
                end_times_dict[event] = float(cleaned_end[i + 1])
            except ValueError:
                end_times_dict[event] = None

    df_event_timings_subject = pd.DataFrame({
        'Event': event_names,
        'Start_Time': [start_times_dict.get(e) for e in event_names],
        'End_Time': [end_times_dict.get(e) for e in event_names]
    })
    # Convert to datetime for easier comparison and alignment
    df_event_timings_subject['Start_Time'] = pd.to_timedelta(df_event_timings_subject['Start_Time'], unit='s')
    df_event_timings_subject['End_Time'] = pd.to_timedelta(df_event_timings_subject['End_Time'], unit='s')

    # --- 2. Resample Chest Data ---
    resampled_chest_data = {}
    for sensor, df in subject_chest_dataframes.items():
        original_freq = CHEST_ACC_ORIG_FREQ if sensor == 'ACC' else CHEST_OTHER_ORIG_FREQ
        resampled_chest_data[sensor] = resample_data(df, original_freq, TARGET_FREQ)
        print(f"  Resampled chest {sensor} from {original_freq}Hz to {TARGET_FREQ}Hz. New shape: {resampled_chest_data[sensor].shape}")

    # --- 3. Resample Wrist Data ---
    resampled_wrist_data = {}
    wrist_orig_freq_map = {'ACC': WRIST_ACC_ORIG_FREQ, 'BVP': WRIST_BVP_ORIG_FREQ, 'EDA': WRIST_EDA_ORIG_FREQ, 'TEMP': WRIST_TEMP_ORIG_FREQ}
    for sensor, df in subject_wrist_dataframes.items():
        original_freq = wrist_orig_freq_map.get(sensor, TARGET_FREQ) # Default to TARGET_FREQ if not found
        resampled_wrist_data[sensor] = resample_data(df, original_freq, TARGET_FREQ)
        print(f"  Resampled wrist {sensor} from {original_freq}Hz to {TARGET_FREQ}Hz. New shape: {resampled_wrist_data[sensor].shape}")

    # --- 4. Process and Align Labels ---
    labels = all_labels_data[subject_name]
    # Labels are originally at 700Hz, we need to resample them to TARGET_FREQ

    # Create a DataFrame for labels to use the resample_data function
    df_labels_orig = pd.DataFrame(labels, columns=['label'])
    df_labels_resampled = resample_data(df_labels_orig, LABEL_ORIG_FREQ, TARGET_FREQ)
    print(f"  Resampled labels from {LABEL_ORIG_FREQ}Hz to {TARGET_FREQ}Hz. New shape: {df_labels_resampled.shape}")

    # --- 5. Consolidate and Align all data for the subject ---
    # Merge all resampled sensor dataframes and labels
    df_subject_merged = pd.DataFrame(index=pd.to_timedelta(np.arange(len(df_labels_resampled)) / float(TARGET_FREQ), unit='s'))

    # Add chest data
    for sensor, df in resampled_chest_data.items():
        # Rename columns to avoid conflicts and indicate origin
        if sensor == 'ACC':
            df_renamed = df.rename(columns={col: f'{sensor}_chest_{col}' for col in df.columns})
        else:
            df_renamed = df.rename(columns={col: f'{sensor}_chest' for col in df.columns})
        df_subject_merged = df_subject_merged.merge(df_renamed, how='outer', left_index=True, right_index=True)

    # Add wrist data
    for sensor, df in resampled_wrist_data.items():
        # Rename columns to avoid conflicts and indicate origin
        if sensor == 'ACC':
            df_renamed = df.rename(columns={col: f'{sensor}_wrist_{col}' for col in df.columns})
        else:
            df_renamed = df.rename(columns={col: f'{sensor}_wrist' for col in df.columns})
        df_subject_merged = df_subject_merged.merge(df_renamed, how='outer', left_index=True, right_index=True)

    # Add labels, ensuring they are named 'label'
    df_subject_merged = df_subject_merged.merge(df_labels_resampled, how='outer', left_index=True, right_index=True)

    # Fill any missing values after outer merge, often due to slight length differences during resampling
    # And also ensuring columns are numerical before applying fillna methods
    for col in df_subject_merged.columns:
        if pd.api.types.is_numeric_dtype(df_subject_merged[col]):
            df_subject_merged[col] = df_subject_merged[col].ffill().bfill()

    # Ensure 'label' column is integer type, as labels are discrete categories.
    # Round to nearest integer before casting, to handle floats introduced by resampling.
    if 'label' in df_subject_merged.columns and pd.api.types.is_float_dtype(df_subject_merged['label']):
        df_subject_merged['label'] = df_subject_merged['label'].round().astype(int)

    # --- Clip data to event timings ---
    # Use the 'Base' event as the primary timing reference for the overall segment
    # Assuming 'Base' is always the first event and 'Medi 2' is always the last relevant event for segmenting
    base_event_start_timedelta = df_event_timings_subject[df_event_timings_subject['Event'] == 'Base']['Start_Time'].iloc[0]
    last_event_end_timedelta = df_event_timings_subject[df_event_timings_subject['Event'] == 'Medi 2']['End_Time'].iloc[0]

    df_subject_final = df_subject_merged[
        (df_subject_merged.index >= base_event_start_timedelta) &
        (df_subject_merged.index <= last_event_end_timedelta)
    ].copy()

    processed_subject_data[subject_name] = df_subject_final
    print(f"  Final processed DataFrame for {subject_name} shape: {df_subject_final.shape}")
    print(f"  Columns for {subject_name}: {df_subject_final.columns.tolist()}")

print("\n--- All subjects processed. ---")
print("Keys in processed_subject_data:", processed_subject_data.keys())

# Display head of one processed subject's data for verification
if processed_subject_data:
    first_subject = list(processed_subject_data.keys())[0]
    print(f"\nHead of {first_subject}'s processed data:")
    print(processed_subject_data[first_subject].head())
    print(f"Info of {first_subject}'s processed data:")
    processed_subject_data[first_subject].info()


Processing data for subjects to 64Hz...

--- Processing S2 ---
  Resampled chest ACC from 700Hz to 64Hz. New shape: (389056, 3)
  Resampled chest ECG from 700Hz to 64Hz. New shape: (389056, 1)
  Resampled chest EMG from 700Hz to 64Hz. New shape: (389056, 1)
  Resampled chest EDA from 700Hz to 64Hz. New shape: (389056, 1)
  Resampled chest Temp from 700Hz to 64Hz. New shape: (389056, 1)
  Resampled chest Resp from 700Hz to 64Hz. New shape: (389056, 1)
  Resampled wrist ACC from 32Hz to 64Hz. New shape: (389056, 3)
  Resampled wrist BVP from 64Hz to 64Hz. New shape: (389056, 1)
  Resampled wrist EDA from 4Hz to 64Hz. New shape: (389056, 1)
  Resampled wrist TEMP from 4Hz to 64Hz. New shape: (389056, 1)
  Resampled labels from 700Hz to 64Hz. New shape: (389056, 1)
  Final processed DataFrame for S2 shape: (5956, 15)
  Columns for S2: ['ACC_chest_ACC_x', 'ACC_chest_ACC_y', 'ACC_chest_ACC_z', 'ECG_chest', 'EMG_chest', 'EDA_chest', 'Temp_chest', 'Resp_chest', 'ACC_wrist_ACC_x', 'ACC_wrist_AC

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# Define parameters for windowing
WINDOW_SIZE = 128  # Number of time steps in each sequence (e.g., 4 seconds at 32Hz)
STEP_SIZE = 64     # Number of time steps to slide the window (e.g., 2 seconds overlap)

def create_windows(data, labels, window_size, step_size):
    X, y = [], []
    for i in range(0, len(data) - window_size, step_size):
        X.append(data[i:i + window_size])
        # For labels, we can take the mode, mean, or the last value in the window
        # For stress detection, often the most frequent label in the window is used
        # Or, if events are distinct, the label at the end of the window
        # Let's take the mode for simplicity for now, handling potential empty modes
        window_labels = labels[i:i+window_size]
        if len(window_labels) > 0:
            modes = pd.Series(window_labels).mode()
            if not modes.empty:
                y.append(modes.iloc[0]) # Take the first mode if multiple
            else:
                y.append(np.nan) # Handle case where no mode can be found (e.g., all nan in window)
        else:
            y.append(np.nan)

    # Filter out windows where label is NaN if they occurred
    valid_indices = ~np.isnan(y)
    X = np.array(X)[valid_indices]
    y = np.array(y)[valid_indices]

    return X, y

# Prepare data for all subjects
X_all, y_all = [], []
subject_scalers = {}

# Iterate through each subject's processed data
for subject_name, df_subject in processed_subject_data.items():
    print(f"\n--- Windowing and Scaling data for {subject_name} ---")

    # Separate features and labels
    features = df_subject.drop(columns=['label'])
    labels = df_subject['label'].values

    # Handle potential NaNs in features due to outer merge and fillna issues
    # For this specific dataset, we expect numerical data. Let's fill any remaining NaNs with 0.
    features = features.fillna(0)

    # Scale features for the current subject
    # It's important to fit the scaler only on the training data later to avoid data leakage
    # However, for now, we'll scale per subject based on their full data for consistency
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)
    subject_scalers[subject_name] = scaler # Store scaler if we need to inverse transform later

    # Create windows for the current subject
    X_subject, y_subject = create_windows(scaled_features, labels, WINDOW_SIZE, STEP_SIZE)

    if X_subject.size > 0:
        X_all.append(X_subject)
        y_all.append(y_subject)
        print(f"  Created {len(X_subject)} windows for {subject_name}")
    else:
        print(f"  No valid windows created for {subject_name}")

# Concatenate all subjects' data
if X_all and y_all:
    X_combined = np.vstack(X_all)
    y_combined = np.concatenate(y_all)
    print(f"\nCombined data from all subjects: X_combined shape {X_combined.shape}, y_combined shape {y_combined.shape}")

    # Split data into training and testing sets
    # We'll need to decide on a robust splitting strategy, e.g., subject-wise split or time-based split
    # For a simple start, let's do a random split on combined windows
    X_train, X_test, y_train, y_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined)

    print(f"\nTraining data shape: {X_train.shape}, {y_train.shape}")
    print(f"Testing data shape: {X_test.shape}, {y_test.shape}")

    # Display unique labels and their counts to check stratification
    unique_labels, counts = np.unique(y_train, return_counts=True)
    print(f"Training label distribution: {dict(zip(unique_labels, counts))}")
    unique_labels, counts = np.unique(y_test, return_counts=True)
    print(f"Testing label distribution: {dict(zip(unique_labels, counts))}")

else:
    print("No data available after processing subjects to create combined dataset.")



In [ ]:
import os

# Remove the existing WESAD_data directory to ensure a clean unzip
if os.path.exists('/content/WESAD_data'):
    !rm -rf /content/WESAD_data
    print("Removed existing /content/WESAD_data directory.")
else:
    print("/content/WESAD_data directory does not exist, proceeding.")


In [ ]:
from google.colab import drive
import os

# 1. Mount the drive (force remount to avoid caching issues)
drive.mount('/content/drive', force_remount=True)

# 2. Define the exact path (Google Drive root is 'MyDrive')
zip_path = '/content/drive/MyDrive/WESAD.zip'

# 3. Check if the file exists before unzipping and perform unzipping without -q
if os.path.exists(zip_path):
    print(f"Found {zip_path}. Unzipping now...")
    # Removed -q flag to see full unzip output and errors
    !unzip "{zip_path}" -d "/content/WESAD_data"
    print("Done! Files are now in the 'WESAD_data' folder.")

    # Verify contents after unzipping
    print('\nListing contents of WESAD_data:')
    !ls -F /content/WESAD_data
    print('\nListing contents of WESAD_data/WESAD (if it exists):')
    if os.path.exists('/content/WESAD_data/WESAD'):
        !ls -F /content/WESAD_data/WESAD
    else:
        print('WESAD subdirectory not found after unzip.')

else:
    print(f"Error: {zip_path} not found. Please ensure the WESAD.zip file is in your main 'My Drive' folder.")


In [ ]:
import pandas as pd
import numpy as np
import os

# Initialize dictionaries to hold individual sensor DataFrames from ALL subjects
all_chest_dataframes = {}
all_wrist_dataframes = {}
all_quest_dataframes = {}
all_labels_data = {} # New: Dictionary to store label data for each subject

# Base path to the WESAD data directory
base_data_path = '/content/WESAD_data/WESAD'
print(f"Base data path: {base_data_path}")
print(f"Does base path exist? {os.path.exists(base_data_path)}")

# Loop through subjects S2 to S17
print("Starting loop through subjects S2-S17...")
for subject_id in range(2, 18):
    subject_name = f'S{subject_id}'
    print(f"Looking for subject: {subject_name}")
    subject_data_path = os.path.join(base_data_path, subject_name)
    pkl_file_path = os.path.join(subject_data_path, f'{subject_name}.pkl')
    quest_file_path = os.path.join(subject_data_path, f'{subject_name}_quest.csv')

    print(f"Checking for pkl file: {pkl_file_path}")
    if os.path.exists(pkl_file_path):
        print(f"Processing {subject_name} data...")
        # Load the .pkl file
        raw_data_dict = pd.read_pickle(pkl_file_path)

        # Process 'chest' signals
        if 'chest' in raw_data_dict['signal']:
            chest_signals = raw_data_dict['signal']['chest']
            for sensor_name, data_array in chest_signals.items():
                key = f'{subject_name}_{sensor_name}'
                if sensor_name == 'ACC':
                    all_chest_dataframes[key] = pd.DataFrame(data_array, columns=['ACC_x', 'ACC_y', 'ACC_z'])
                else:
                    all_chest_dataframes[key] = pd.DataFrame(data_array.flatten(), columns=[sensor_name])

        # Process 'wrist' signals
        if 'wrist' in raw_data_dict['signal']:
            wrist_signals = raw_data_dict['signal']['wrist']
            for sensor_name, data_array in wrist_signals.items():
                key = f'{subject_name}_{sensor_name}'
                if sensor_name == 'ACC':
                    all_wrist_dataframes[key] = pd.DataFrame(data_array, columns=['ACC_x', 'ACC_y', 'ACC_z'])
                else:
                    all_wrist_dataframes[key] = pd.DataFrame(data_array.flatten(), columns=[sensor_name])

        # Extract labels
        if 'label' in raw_data_dict:
            all_labels_data[subject_name] = raw_data_dict['label']
            print(f"Loaded labels for {subject_name}")

        # Load the _quest.csv file
        if os.path.exists(quest_file_path):
            all_quest_dataframes[subject_name] = pd.read_csv(quest_file_path)
            print(f"Loaded {subject_name}_quest.csv")
        else:
            print(f"Warning: {subject_name}_quest.csv not found.")

    else:
        print(f"Warning: {subject_name}.pkl not found at {pkl_file_path}")

print("\nFinished loading data for all subjects.")
print("Total chest dataframes loaded:", len(all_chest_dataframes))
print("Total wrist dataframes loaded:", len(all_wrist_dataframes))
print("Total quest dataframes loaded:", len(all_quest_dataframes))
print("Total labels data loaded:", len(all_labels_data))

# For compatibility with subsequent cells that perform initial inspection and cleaning,
# these assignments are kept. The `chest_dataframes` and `wrist_dataframes` will be
# modified (deduplicated) by later cells.
chest_dataframes = all_chest_dataframes
wrist_dataframes = all_wrist_dataframes

# Set df_s2_quest to S10's quest data if S2 is not available for compatibility with
# existing cells that extract event timings or questionnaire responses.
if 'S2' in all_quest_dataframes:
    df_s2_quest = all_quest_dataframes['S2']
elif 'S10' in all_quest_dataframes:
    df_s2_quest = all_quest_dataframes['S10']
    print("Using S10 quest data as a fallback for df_s2_quest due to missing S2 data.")
else:
    df_s2_quest = None # Or handle error if S2 quest is essential

# Define s2_data_path for compatibility, defaulting to S10 if S2 is missing
if os.path.exists(os.path.join(base_data_path, 'S2')):
    s2_data_path = os.path.join(base_data_path, 'S2')
elif os.path.exists(os.path.join(base_data_path, 'S10')):
    s2_data_path = os.path.join(base_data_path, 'S10')
    print("Using S10 data path as a fallback for s2_data_path due to missing S2 data.")
else:
    s2_data_path = None


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!ls "/content/drive/MyDrive/WESAD.zip"


In [ ]:
print('Listing contents of MyDrive:')
!ls "/content/drive/MyDrive"


In [ ]:
from google.colab import drive
import os

# 1. Mount the drive
drive.mount('/content/drive')

# 2. Define the exact path (Google Drive root is 'MyDrive')
zip_path = '/content/drive/MyDrive/WESAD.zip'

# 3. Check if the file exists before unzipping
if os.path.exists(zip_path):
    print("Found it! Unzipping now...")
    !unzip -q "{zip_path}" -d "/content/WESAD_data"
    print("Done! Files are now in the 'WESAD_data' folder on the left sidebar.")
else:
    print("Still can't see it. Make sure the file is in the main 'My Drive' folder, not a subfolder.")


In [ ]:
print('Listing contents of the unzipped WESAD data directory:')
!ls -F /content/WESAD_data


In [ ]:
print('Listing contents of the WESAD subdirectory:')
!ls -F /content/WESAD_data/WESAD


In [ ]:
print('Listing contents of the S2 subdirectory:')
!ls -F /content/WESAD_data/WESAD/S2


In [ ]:
import pandas as pd

# Define the base path to the S2 subject data directory
s2_data_path = '/content/WESAD_data/WESAD/S2'

# Load the S2.pkl file into a pandas DataFrame
df_s2_data = pd.read_pickle(f'{s2_data_path}/S2.pkl')

# Load the S2_quest.csv file into a pandas DataFrame
df_s2_quest = pd.read_csv(f'{s2_data_path}/S2_quest.csv')

print("df_s2_data head:")
print(df_s2_data.head())

print("\ndf_s2_quest head:")
print(df_s2_quest.head())


In [ ]:
import pandas as pd

# Define the base path to the S2 subject data directory
s2_data_path = '/content/WESAD_data/WESAD/S2'

# Load the S2.pkl file into a dictionary first
s2_raw_data_dict = pd.read_pickle(f'{s2_data_path}/S2.pkl')

# Extract 'chest' and 'wrist' signals and convert them to DataFrames
df_s2_chest_data = pd.DataFrame(s2_raw_data_dict['signal']['chest'])
df_s2_wrist_data = pd.DataFrame(s2_raw_data_dict['signal']['wrist'])

# Load the S2_quest.csv file into a pandas DataFrame
df_s2_quest = pd.read_csv(f'{s2_data_path}/S2_quest.csv')

print("df_s2_chest_data head:")
print(df_s2_chest_data.head())

print("\ndf_s2_wrist_data head:")
print(df_s2_wrist_data.head())

print("\ndf_s2_quest head:")
print(df_s2_quest.head())


In [ ]:
import pandas as pd
import numpy as np

# Define the base path to the S2 subject data directory
s2_data_path = '/content/WESAD_data/WESAD/S2'

# Load the S2.pkl file into a dictionary first
s2_raw_data_dict = pd.read_pickle(f'{s2_data_path}/S2.pkl')

# Process 'chest' signals
chest_signals = s2_raw_data_dict['signal']['chest']
chest_data = {}
for sensor_name, data_array in chest_signals.items():
    if sensor_name == 'ACC':
        # Accelerometer data is typically 3-axis, so create three columns
        chest_data['ACC_x'] = data_array[:, 0]
        chest_data['ACC_y'] = data_array[:, 1]
        chest_data['ACC_z'] = data_array[:, 2]
    else:
        # Other sensors are typically 1D
        chest_data[sensor_name] = data_array

df_s2_chest_data = pd.DataFrame(chest_data)

# Process 'wrist' signals
wrist_signals = s2_raw_data_dict['signal']['wrist']
wrist_data = {}
for sensor_name, data_array in wrist_signals.items():
    if sensor_name == 'ACC':
        # Accelerometer data is typically 3-axis, so create three columns
        wrist_data['ACC_x'] = data_array[:, 0]
        wrist_data['ACC_y'] = data_array[:, 1]
        wrist_data['ACC_z'] = data_array[:, 2]
    else:
        # Other sensors are typically 1D
        wrist_data[sensor_name] = data_array

df_s2_wrist_data = pd.DataFrame(wrist_data)

# Load the S2_quest.csv file into a pandas DataFrame
df_s2_quest = pd.read_csv(f'{s2_data_path}/S2_quest.csv')

print("df_s2_chest_data head:")
print(df_s2_chest_data.head())

print("\ndf_s2_wrist_data head:")
print(df_s2_wrist_data.head())

print("\ndf_s2_quest head:")
print(df_s2_quest.head())


In [ ]:
import pandas as pd
import numpy as np

# Define the base path to the S2 subject data directory
s2_data_path = '/content/WESAD_data/WESAD/S2'

# Load the S2.pkl file into a dictionary first
s2_raw_data_dict = pd.read_pickle(f'{s2_data_path}/S2.pkl')

# Process 'chest' signals
chest_signals = s2_raw_data_dict['signal']['chest']
chest_data = {}
for sensor_name, data_array in chest_signals.items():
    if sensor_name == 'ACC':
        # Accelerometer data is typically 3-axis, so create three columns
        chest_data['ACC_x'] = data_array[:, 0]
        chest_data['ACC_y'] = data_array[:, 1]
        chest_data['ACC_z'] = data_array[:, 2]
    else:
        # Other sensors might be 2D arrays with a single column, flatten them
        chest_data[sensor_name] = data_array.flatten()

df_s2_chest_data = pd.DataFrame(chest_data)

# Process 'wrist' signals
wrist_signals = s2_raw_data_dict['signal']['wrist']
wrist_data = {}
for sensor_name, data_array in wrist_signals.items():
    if sensor_name == 'ACC':
        # Accelerometer data is typically 3-axis, so create three columns
        wrist_data['ACC_x'] = data_array[:, 0]
        wrist_data['ACC_y'] = data_array[:, 1]
        wrist_data['ACC_z'] = data_array[:, 2]
    else:
        # Other sensors might be 2D arrays with a single column, flatten them
        wrist_data[sensor_name] = data_array.flatten()

df_s2_wrist_data = pd.DataFrame(wrist_data)

# Load the S2_quest.csv file into a pandas DataFrame
df_s2_quest = pd.read_csv(f'{s2_data_path}/S2_quest.csv')

print("df_s2_chest_data head:")
print(df_s2_chest_data.head())

print("\ndf_s2_wrist_data head:")
print(df_s2_wrist_data.head())

print("\ndf_s2_quest head:")
print(df_s2_quest.head())


In [ ]:
import pandas as pd
import numpy as np

# Define the base path to the S2 subject data directory
s2_data_path = '/content/WESAD_data/WESAD/S2'

# Load the S2.pkl file into a dictionary first
s2_raw_data_dict = pd.read_pickle(f'{s2_data_path}/S2.pkl')

# Initialize dictionaries to hold individual sensor DataFrames
chest_dataframes = {}
wrist_dataframes = {}

# Process 'chest' signals
chest_signals = s2_raw_data_dict['signal']['chest']
for sensor_name, data_array in chest_signals.items():
    if sensor_name == 'ACC':
        # Accelerometer data is 3-axis, create three columns
        chest_dataframes[sensor_name] = pd.DataFrame(data_array, columns=['ACC_x', 'ACC_y', 'ACC_z'])
    else:
        # Other sensors are typically 1D or 2D with one column, flatten and create a DataFrame
        chest_dataframes[sensor_name] = pd.DataFrame(data_array.flattenbase_data_path = '/content/WESAD_data/' # Corrected path
         (), columns=[sensor_name])

# Process 'wrist' signals
wrist_signals = s2_raw_data_dict['signal']['wrist']
for sensor_name, data_array in wrist_signals.items():
    if sensor_name == 'ACC':
        # Accelerometer data is 3-axis, create three columns
        wrist_dataframes[sensor_name] = pd.DataFrame(data_array, columns=['ACC_x', 'ACC_y', 'ACC_z'])
    else:
        # Other sensors are typically 1D or 2D with one column, flatten and create a DataFrame
        wrist_dataframes[sensor_name] = pd.DataFrame(data_array.flatten(), columns=[sensor_name])

# Load the S2_quest.csv file into a pandas DataFrame
df_s2_quest = pd.read_csv(f'{s2_data_path}/S2_quest.csv')

print("Chest DataFrames (first 5 rows of each):")
for sensor_name, df in chest_dataframes.items():
    print(f"\n{sensor_name}:")
    print(df.head())

print("\nWrist DataFrames (first 5 rows of each):")
for sensor_name, df in wrist_dataframes.items():
    print(f"\n{sensor_name}:")
    print(df.head())

print("\ndf_s2_quest head:")
print(df_s2_quest.head())


In [ ]:
import pandas as pd
import numpy as np
import os

# Initialize dictionaries to hold individual sensor DataFrames from ALL subjects
all_chest_dataframes = {}
all_wrist_dataframes = {}
all_quest_dataframes = {}
all_labels_data = {} # New: Dictionary to store label data for each subject

# Base path to the WESAD data directory
base_data_path = '/content/WESAD_data/WESAD'
print(f"Base data path: {base_data_path}")
print(f"Does base path exist? {os.path.exists(base_data_path)}")

# Loop through subjects S2 to S17
print("Starting loop through subjects S2-S17...")
for subject_id in range(2, 18):
    subject_name = f'S{subject_id}'
    print(f"Looking for subject: {subject_name}")
    subject_data_path = os.path.join(base_data_path, subject_name)
    pkl_file_path = os.path.join(subject_data_path, f'{subject_name}.pkl')
    quest_file_path = os.path.join(subject_data_path, f'{subject_name}_quest.csv')

    print(f"Checking for pkl file: {pkl_file_path}")
    if os.path.exists(pkl_file_path):
        print(f"Processing {subject_name} data...")
        # Load the .pkl file
        raw_data_dict = pd.read_pickle(pkl_file_path)

        # Process 'chest' signals
        if 'chest' in raw_data_dict['signal']:
            chest_signals = raw_data_dict['signal']['chest']
            for sensor_name, data_array in chest_signals.items():
                key = f'{subject_name}_{sensor_name}'
                if sensor_name == 'ACC':
                    all_chest_dataframes[key] = pd.DataFrame(data_array, columns=['ACC_x', 'ACC_y', 'ACC_z'])
                else:
                    all_chest_dataframes[key] = pd.DataFrame(data_array.flatten(), columns=[sensor_name])

        # Process 'wrist' signals
        if 'wrist' in raw_data_dict['signal']:
            wrist_signals = raw_data_dict['signal']['wrist']
            for sensor_name, data_array in wrist_signals.items():
                key = f'{subject_name}_{sensor_name}'
                if sensor_name == 'ACC':
                    all_wrist_dataframes[key] = pd.DataFrame(data_array, columns=['ACC_x', 'ACC_y', 'ACC_z'])
                else:
                    all_wrist_dataframes[key] = pd.DataFrame(data_array.flatten(), columns=[sensor_name])

        # Extract labels
        if 'label' in raw_data_dict:
            all_labels_data[subject_name] = raw_data_dict['label']
            print(f"Loaded labels for {subject_name}")

        # Load the _quest.csv file
        if os.path.exists(quest_file_path):
            all_quest_dataframes[subject_name] = pd.read_csv(quest_file_path)
            print(f"Loaded {subject_name}_quest.csv")
        else:
            print(f"Warning: {subject_name}_quest.csv not found.")

    else:
        print(f"Warning: {subject_name}.pkl not found at {pkl_file_path}")

print("\nFinished loading data for all subjects.")
print("Total chest dataframes loaded:", len(all_chest_dataframes))
print("Total wrist dataframes loaded:", len(all_wrist_dataframes))
print("Total quest dataframes loaded:", len(all_quest_dataframes))
print("Total labels data loaded:", len(all_labels_data))

# For compatibility with subsequent cells that perform initial inspection and cleaning,
# these assignments are kept. The `chest_dataframes` and `wrist_dataframes` will be
# modified (deduplicated) by later cells.
chest_dataframes = all_chest_dataframes
wrist_dataframes = all_wrist_dataframes

# Set df_s2_quest to S10's quest data if S2 is not available for compatibility with
# existing cells that extract event timings or questionnaire responses.
if 'S2' in all_quest_dataframes:
    df_s2_quest = all_quest_dataframes['S2']
elif 'S10' in all_quest_dataframes:
    df_s2_quest = all_quest_dataframes['S10']
    print("Using S10 quest data as a fallback for df_s2_quest due to missing S2 data.")
else:
    df_s2_quest = None # Or handle error if S2 quest is essential

# Define s2_data_path for compatibility, defaulting to S10 if S2 is missing
if os.path.exists(os.path.join(base_data_path, 'S2')):
    s2_data_path = os.path.join(base_data_path, 'S2')
elif os.path.exists(os.path.join(base_data_path, 'S10')):
    s2_data_path = os.path.join(base_data_path, 'S10')
    print("Using S10 data path as a fallback for s2_data_path due to missing S2 data.")
else:
    s2_data_path = None


Base data path: /content/WESAD_data/WESAD
Does base path exist? True
Starting loop through subjects S2-S17...
Looking for subject: S2
Checking for pkl file: /content/WESAD_data/WESAD/S2/S2.pkl
Processing S2 data...
Loaded labels for S2
Loaded S2_quest.csv
Looking for subject: S3
Checking for pkl file: /content/WESAD_data/WESAD/S3/S3.pkl
Processing S3 data...
Loaded labels for S3
Loaded S3_quest.csv
Looking for subject: S4
Checking for pkl file: /content/WESAD_data/WESAD/S4/S4.pkl
Processing S4 data...
Loaded labels for S4
Loaded S4_quest.csv
Looking for subject: S5
Checking for pkl file: /content/WESAD_data/WESAD/S5/S5.pkl
Processing S5 data...
Loaded labels for S5
Loaded S5_quest.csv
Looking for subject: S6
Checking for pkl file: /content/WESAD_data/WESAD/S6/S6.pkl
Processing S6 data...
Loaded labels for S6
Loaded S6_quest.csv
Looking for subject: S7
Checking for pkl file: /content/WESAD_data/WESAD/S7/S7.pkl
Processing S7 data...
Loaded labels for S7
Loaded S7_quest.csv
Looking for su

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import resample

# Target frequency for resampling
TARGET_FREQ = 32  # Hz
TOLERANCE = '10ms' # Tolerance for nearest matching

# --- Resample Chest Data (Downsampling) ---
resampled_chest_dataframes = {}
# Iterate through the globally available chest_dataframes (which now contain S10/S11 data)
for sensor_name_key, df in chest_dataframes.items():
    # Only process S10 data if S2 is not available
    if 'S2' in sensor_name_key and not os.path.exists(os.path.join(base_data_path, 'S2')):
        continue # Skip S2 if it's not present

    # Adapt sensor_name to be just the sensor type (e.g., 'ACC' from 'S10_ACC')
    actual_sensor_name = sensor_name_key.split('_')[1] if '_' in sensor_name_key else sensor_name_key

    if not df.empty and actual_sensor_name == 'ACC': # ACC is 700Hz
        original_len = len(df)
        if original_len > 1:
            time_index = pd.to_timedelta(np.arange(original_len) / 700.0, unit='s')
            df.index = time_index
            resample_len = int(original_len * TARGET_FREQ / 700.0)
            resampled_data = {}
            for col in df.columns:
                resampled_data[col] = resample(df[col], resample_len)
            resampled_df = pd.DataFrame(resampled_data)
            resampled_time_index = pd.to_timedelta(np.arange(resample_len) / float(TARGET_FREQ), unit='s')
            resampled_df.index = resampled_time_index
            resampled_chest_dataframes[sensor_name_key] = resampled_df
            print(f"Resampled {sensor_name_key} from {original_len} to {resample_len} points (700Hz to {TARGET_FREQ}Hz)")
        else:
            resampled_chest_dataframes[sensor_name_key] = df.copy()
            print(f"Kept {sensor_name_key} as is (empty or single row)")
    else:
        resampled_chest_dataframes[sensor_name_key] = df.copy()
        if not df.empty:
             original_len = len(df)
             time_index = pd.to_timedelta(np.arange(original_len) / float(TARGET_FREQ), unit='s')
             df.index = time_index
             resampled_chest_dataframes[sensor_name_key].index = time_index
             print(f"Kept non-ACC {sensor_name_key} with {original_len} points, assumed {TARGET_FREQ}Hz")
        else:
            print(f"Kept non-ACC {sensor_name_key} as is (empty)")

# --- Resample Wrist Data (Upsampling/Downsampling) ---
resampled_wrist_dataframes = {}
wrist_original_freqs = {'ACC': 32, 'BVP': 64, 'EDA': 4, 'TEMP': 4}

# Iterate through the globally available wrist_dataframes (which now contain S10/S11 data)
for sensor_name_key, df in wrist_dataframes.items():
    # Only process S10 data if S2 is not available
    if 'S2' in sensor_name_key and not os.path.exists(os.path.join(base_data_path, 'S2')):
        continue # Skip S2 if it's not present

    actual_sensor_name = sensor_name_key.split('_')[1] if '_' in sensor_name_key else sensor_name_key

    if not df.empty and actual_sensor_name in wrist_original_freqs:
        original_freq = wrist_original_freqs[actual_sensor_name]
        original_len = len(df)
        if original_len > 1 and original_freq != TARGET_FREQ:
            time_index = pd.to_timedelta(np.arange(original_len) / float(original_freq), unit='s')
            df.index = time_index
            resample_len = int(original_len * TARGET_FREQ / float(original_freq))
            resampled_data = {}
            for col in df.columns:
                resampled_data[col] = resample(df[col], resample_len)
            resampled_df = pd.DataFrame(resampled_data)
            resampled_time_index = pd.to_timedelta(np.arange(resample_len) / float(TARGET_FREQ), unit='s')
            resampled_df.index = resampled_time_index
            resampled_wrist_dataframes[sensor_name_key] = resampled_df
            print(f"Resampled {sensor_name_key} from {original_len} to {resample_len} points ({original_freq}Hz to {TARGET_FREQ}Hz)")
        else:
            if original_len > 0:
                time_index = pd.to_timedelta(np.arange(original_len) / float(original_freq), unit='s')
                df.index = time_index
                resampled_wrist_dataframes[sensor_name_key] = df.copy()
                print(f"Kept {sensor_name_key} as is with {original_len} points (at/near {TARGET_FREQ}Hz or single row)")
            else:
                resampled_wrist_dataframes[sensor_name_key] = df.copy()
                print(f"Kept {sensor_name_key} as is (empty)")
    else:
        resampled_wrist_dataframes[sensor_name_key] = df.copy()
        print(f"Kept {sensor_name_key} as is (empty or not in freq map)")

print("\nResampling complete.")

# --- Alignment ---
# The following section expects df_event_timings which is created from df_s2_quest
# We will use df_s2_quest (which might be S10's quest data as a fallback) to create df_event_timings
if 'df_event_timings' not in locals() and 'df_event_timings' not in globals() and df_s2_quest is not None:
    print("Creating df_event_timings from available df_s2_quest...")
    # 1. Locate and extract the string content from the relevant rows
    order_str = df_s2_quest.iloc[0, 0]
    start_str = df_s2_quest.iloc[1, 0]
    end_str = df_s2_quest.iloc[2, 0]

    # 2. Split by semicolon and clean the lists
    def clean_split_list(s):
        parts = s.replace('#', '').split(';')
        return [p.strip() for p in parts if p.strip()]

    cleaned_order = clean_split_list(order_str)
    cleaned_start = clean_split_list(start_str)
    cleaned_end = clean_split_list(end_str)

    # 3. Create a list of event names (skipping the 'ORDER' label)
    event_names = cleaned_order[1:]

    # 4. Create dictionaries for START and END times, converting to float
    start_times = {}
    for i, event in enumerate(event_names):
        if (i + 1) < len(cleaned_start):
            try:
                start_times[event] = float(cleaned_start[i + 1])
            except ValueError:
                start_times[event] = None

    end_times = {}
    for i, event in enumerate(event_names):
        if (i + 1) < len(cleaned_end):
            try:
                end_times[event] = float(cleaned_end[i + 1])
            except ValueError:
                end_times[event] = None

    # 5. Combine into a new pandas DataFrame
    events_list = []
    start_time_list = []
    end_time_list = []

    for event in event_names:
        events_list.append(event)
        start_time_list.append(start_times.get(event))
        end_time_list.append(end_times.get(event))

    df_event_timings = pd.DataFrame({
        'Event': events_list,
        'Start_Time': start_time_list,
        'End_Time': end_time_list
    })
    print("df_event_timings created successfully.")
elif df_s2_quest is None:
    print("Warning: df_s2_quest is None, cannot create df_event_timings. Using dummy values for alignment.")
    start_time = pd.to_datetime('1970-01-01 00:00:00') # Dummy start
    end_time = pd.to_datetime('1970-01-01 00:05:00')   # Dummy end (5 mins later)
    df_event_timings = pd.DataFrame({'Start_Time': [start_time], 'End_Time': [end_time]})
else:
    print("df_event_timings already exists.")

# Ensure Start_Time and End_Time are datetime objects for alignment
df_event_timings['Start_Time'] = pd.to_datetime(df_event_timings['Start_Time'], unit='s', origin='unix')
df_event_timings['End_Time'] = pd.to_datetime(df_event_timings['End_Time'], unit='s', origin='unix')

start_time = df_event_timings['Start_Time'].iloc[0]
end_time = df_event_timings['End_Time'].iloc[0]

aligned_data = {}

# Align resampled chest data
for sensor_name_key, df in resampled_chest_dataframes.items():
    if not df.empty and isinstance(df.index, pd.TimedeltaIndex):
        df_abs_time = df.copy()
        df_abs_time.index = start_time + df_abs_time.index
        aligned_df = df_abs_time[(df_abs_time.index >= start_time) & (df_abs_time.index <= end_time)]
        # Make index relative to start_time again for merging
        aligned_df.index = aligned_df.index - start_time
        aligned_data[f'chest_{sensor_name_key}'] = aligned_df
        print(f"Aligned chest_{sensor_name_key}")

# Align resampled wrist data
for sensor_name_key, df in resampled_wrist_dataframes.items():
    if not df.empty and isinstance(df.index, pd.TimedeltaIndex):
        df_abs_time = df.copy()
        df_abs_time.index = start_time + df_abs_time.index
        aligned_df = df_abs_time[(df_abs_time.index >= start_time) & (df_abs_time.index <= end_time)]
        aligned_df.index = aligned_df.index - start_time
        aligned_data[f'wrist_{sensor_name_key}'] = aligned_df
        print(f"Aligned wrist_{sensor_name_key}")

# --- Consolidation ---
df_final = pd.DataFrame()
for sensor_key, df in aligned_data.items():
    if not df.empty:
        # Ensure unique column names by prepending sensor_key to original column names
        df_renamed = df.rename(columns={col: f'{sensor_key}_{col}' for col in df.columns})
        if df_final.empty:
            df_final = df_renamed
        else:
            # Before merging, ensure indexes are unique or handle potential duplicates if they arise
            # For now, relying on merge_asof handling potential non-exact index matches
            df_final = pd.merge_asof(df_final.sort_index(), df_renamed.sort_index(), left_index=True, right_index=True, direction='nearest', tolerance=pd.Timedelta(TOLERANCE))

print("\nAlignment and Consolidation complete.")
print("Final DataFrame head:")
print(df_final.head())
print("Final DataFrame info:")
df_final.info()


In [ ]:
print("--- Initial Data Inspection for Chest Sensor DataFrames ---")
for sensor_name, df in chest_dataframes.items():
    print(f"\n----- {sensor_name} Data (Chest) -----")
    print(f"\n{sensor_name}.head():")
    print(df.head())
    print(f"\n{sensor_name}.info():")
    df.info()
    print(f"\n{sensor_name}.describe():")
    print(df.describe())

print("\n--- Initial Data Inspection for Wrist Sensor DataFrames ---")
for sensor_name, df in wrist_dataframes.items():
    print(f"\n----- {sensor_name} Data (Wrist) -----")
    print(f"\n{sensor_name}.head():")
    print(df.head())
    print(f"\n{sensor_name}.info():")
    df.info()
    print(f"\n{sensor_name}.describe():")
    print(df.describe())

print("\n--- Initial Data Inspection for Questionnaire Data (df_s2_quest) ---")
print("\ndf_s2_quest.head():")
print(df_s2_quest.head())
print("\ndf_s2_quest.info():")
df_s2_quest.info()
print("\ndf_s2_quest.describe():")
print(df_s2_quest.describe())


In [ ]:
print("--- Checking for Missing Values in Chest Sensor DataFrames ---")
for sensor_name, df in chest_dataframes.items():
    print(f"\nMissing values in {sensor_name} (Chest):")
    print(df.isnull().sum())

print("\n--- Checking for Missing Values in Wrist Sensor DataFrames ---")
for sensor_name, df in wrist_dataframes.items():
    print(f"\nMissing values in {sensor_name} (Wrist):")
    print(df.isnull().sum())


```markdown
**Regarding `df_s2_quest`:**

The `df_s2_quest` DataFrame is not yet in a tabular format suitable for a direct `.isnull().sum()` check. It contains metadata and questionnaire responses within a few rows, separated by semicolons. This DataFrame will require further parsing and cleaning before a meaningful assessment of missing values can be performed for individual questions/fields.

Based on the current state of the chest and wrist sensor data, there are no missing values detected.
```

In [ ]:
print("--- Checking for Duplicate Rows in Chest Sensor DataFrames ---")
for sensor_name, df in chest_dataframes.items():
    print(f"\nDuplicate rows in {sensor_name} (Chest): {df.duplicated().sum()}")

print("\n--- Checking for Duplicate Rows in Wrist Sensor DataFrames ---")
for sensor_name, df in wrist_dataframes.items():
    print(f"\nDuplicate rows in {sensor_name} (Wrist): {df.duplicated().sum()}")


```markdown
**Regarding `df_s2_quest`:**

The `df_s2_quest` DataFrame is not yet in a tabular format suitable for a direct `.duplicated().sum()` check. It contains metadata and questionnaire responses within a few rows, separated by semicolons. This DataFrame will require further parsing and cleaning before a meaningful assessment of duplicate rows can be performed for individual questions/fields.

**Regarding Sensor DataFrames:**

Many of the chest and wrist sensor DataFrames contain a significant number of duplicate rows. This will need to be addressed in the next step to ensure data integrity and prevent skewed analysis.
```

In [ ]:
print("--- Removing Duplicate Rows from Chest Sensor DataFrames ---")
for sensor_name, df in chest_dataframes.items():
    initial_duplicates = df.duplicated().sum()
    if initial_duplicates > 0:
        chest_dataframes[sensor_name] = df.drop_duplicates().reset_index(drop=True)
        print(f"Removed {initial_duplicates} duplicate rows from {sensor_name} (Chest). New count: {chest_dataframes[sensor_name].duplicated().sum()}")
    else:
        print(f"No duplicate rows found in {sensor_name} (Chest).")

print("\n--- Removing Duplicate Rows from Wrist Sensor DataFrames ---")
for sensor_name, df in wrist_dataframes.items():
    initial_duplicates = df.duplicated().sum()
    if initial_duplicates > 0:
        wrist_dataframes[sensor_name] = df.drop_duplicates().reset_index(drop=True)
        print(f"Removed {initial_duplicates} duplicate rows from {sensor_name} (Wrist). New count: {wrist_dataframes[sensor_name].duplicated().sum()}")
    else:
        print(f"No duplicate rows found in {sensor_name} (Wrist).")


In [ ]:
print("--- Checking Data Types for Chest Sensor DataFrames ---")
for sensor_name, df in chest_dataframes.items():
    print(f"\n----- {sensor_name} Data (Chest) -----")
    df.info()

print("\n--- Checking Data Types for Wrist Sensor DataFrames ---")
for sensor_name, df in wrist_dataframes.items():
    print(f"\n----- {sensor_name} Data (Wrist) -----")
    df.info()

print("\n--- Checking Data Types for Questionnaire Data (df_s2_quest) ---")
df_s2_quest.info()


In [ ]:
import os

# Define the directory to save cleaned data
cleaned_data_dir = f'{s2_data_path}/cleaned_data'

# Create the directory if it does not exist
os.makedirs(cleaned_data_dir, exist_ok=True)

print(f"Saving cleaned data to: {cleaned_data_dir}")

# Save df_event_timings
df_event_timings.to_csv(f'{cleaned_data_dir}/df_event_timings.csv', index=False)
print("Saved df_event_timings.csv")

# Save df_questionnaire_responses
df_questionnaire_responses.to_csv(f'{cleaned_data_dir}/df_questionnaire_responses.csv', index=False)
print("Saved df_questionnaire_responses.csv")

# Save individual chest sensor DataFrames
for sensor_name, df in chest_dataframes.items():
    df.to_csv(f'{cleaned_data_dir}/chest_{sensor_name.lower()}.csv', index=False)
    print(f"Saved chest_{sensor_name.lower()}.csv")

# Save individual wrist sensor DataFrames
for sensor_name, df in wrist_dataframes.items():
    df.to_csv(f'{cleaned_data_dir}/wrist_{sensor_name.lower()}.csv', index=False)
    print(f"Saved wrist_{sensor_name.lower()}.csv")

print("All cleaned DataFrames saved successfully!")


In [ ]:
print("--- Displaying all rows of df_s2_quest ---")
print(df_s2_quest.to_string())

print("\n--- First row of df_s2_quest ---")
print(df_s2_quest.iloc[0])

print("\n--- Second row of df_s2_quest ---")
print(df_s2_quest.iloc[1])

print("\n--- Third row of df_s2_quest ---")
print(df_s2_quest.iloc[2])

print("\n--- Fifth row of df_s2_quest (index 4) ---")
print(df_s2_quest.iloc[4])


In [ ]:
import pandas as pd

# 1. Locate and extract the string content from the relevant rows
order_str = df_s2_quest.iloc[0, 0]
start_str = df_s2_quest.iloc[1, 0]
end_str = df_s2_quest.iloc[2, 0]

# 2. Split by semicolon and clean the lists
# Remove empty strings and the initial '#' from the first element

def clean_split_list(s):
    # Remove the first '#' if present and split
    parts = s.replace('#', '').split(';')
    # Filter out empty strings
    return [p.strip() for p in parts if p.strip()]

cleaned_order = clean_split_list(order_str)
cleaned_start = clean_split_list(start_str)
cleaned_end = clean_split_list(end_str)

# 3. Create a list of event names (skipping the 'ORDER' label)
event_names = cleaned_order[1:]

# 4. Create dictionaries for START and END times, converting to float
start_times = {}
for i, event in enumerate(event_names):
    # Ensure index is within bounds for start_str data, skipping the 'START' label
    if (i + 1) < len(cleaned_start):
        try:
            start_times[event] = float(cleaned_start[i + 1])
        except ValueError:
            start_times[event] = None # Handle cases where conversion to float fails

end_times = {}
for i, event in enumerate(event_names):
    # Ensure index is within bounds for end_str data, skipping the 'END' label
    if (i + 1) < len(cleaned_end):
        try:
            end_times[event] = float(cleaned_end[i + 1])
        except ValueError:
            end_times[event] = None # Handle cases where conversion to float fails

# 5. Combine into a new pandas DataFrame
# Create lists for DataFrame construction
events_list = []
start_time_list = []
end_time_list = []

for event in event_names:
    events_list.append(event)
    start_time_list.append(start_times.get(event))
    end_time_list.append(end_times.get(event))

df_event_timings = pd.DataFrame({
    'Event': events_list,
    'Start_Time': start_time_list,
    'End_Time': end_time_list
})

print("Event Timings DataFrame:")
print(df_event_timings)


In [ ]:
import pandas as pd

# Initialize empty lists to store parsed responses for each questionnaire type
panas_data = []
stai_data = []
dim_data = []
sssq_data = []

# Iterate through df_s2_quest from row index 4 onwards (as metadata is already extracted)
for i in range(4, len(df_s2_quest)):
    row_string = df_s2_quest.iloc[i, 0]

    # Check if the row starts with a questionnaire identifier and process it
    if row_string.startswith('# PANAS'):
        # Remove the tag and split by semicolon, then convert to numeric
        parts = row_string.replace('# PANAS;', '').split(';')
        numeric_parts = [float(p.strip()) for p in parts if p.strip()]
        if numeric_parts:
            panas_data.append(numeric_parts)
    elif row_string.startswith('# STAI'):
        parts = row_string.replace('# STAI;', '').split(';')
        numeric_parts = [float(p.strip()) for p in parts if p.strip()]
        if numeric_parts:
            stai_data.append(numeric_parts)
    elif row_string.startswith('# DIM'):
        parts = row_string.replace('# DIM;', '').split(';')
        numeric_parts = [float(p.strip()) for p in parts if p.strip()]
        if numeric_parts:
            dim_data.append(numeric_parts)
    elif row_string.startswith('# SSSQ'):
        parts = row_string.replace('# SSSQ;', '').split(';')
        numeric_parts = [float(p.strip()) for p in parts if p.strip()]
        if numeric_parts:
            sssq_data.append(numeric_parts)

# Convert lists of lists to pandas DataFrames. pd.DataFrame handles varying row lengths by padding with NaN.
df_panas = pd.DataFrame(panas_data)
df_stai = pd.DataFrame(stai_data)
df_dim = pd.DataFrame(dim_data)
df_sssq = pd.DataFrame(sssq_data)

# Print the head of each created DataFrame to inspect the parsed questionnaire responses
print("\n--- df_panas head ---")
print(df_panas.head())

print("\n--- df_stai head ---")
print(df_stai.head())

print("\n--- df_dim head ---")
print(df_dim.head())

print("\n--- df_sssq head ---")
print(df_sssq.head())


In [ ]:
import numpy as np

# 1. Rename the columns of each DataFrame
df_panas.columns = [f'PANAS_{i}' for i in range(len(df_panas.columns))]
df_stai.columns = [f'STAI_{i}' for i in range(len(df_stai.columns))]
df_dim.columns = [f'DIM_{i}' for i in range(len(df_dim.columns))]
df_sssq.columns = [f'SSSQ_{i}' for i in range(len(df_sssq.columns))]

# 2. Concatenate df_panas, df_stai, and df_dim horizontally
df_combined_responses = pd.concat([df_panas, df_stai, df_dim], axis=1)

# 3. Create a new DataFrame for df_sssq with the same number of rows as df_combined_responses
# Get the number of rows from df_combined_responses
num_rows = len(df_combined_responses)

# Create an empty DataFrame with the correct number of rows and columns from df_sssq
df_sssq_extended = pd.DataFrame(np.nan, index=range(num_rows), columns=df_sssq.columns)

# Place the actual df_sssq data (which is a single row) into the first row of the extended DataFrame
if not df_sssq.empty:
    df_sssq_extended.iloc[0] = df_sssq.iloc[0]

# 4. Horizontally concatenate the extended df_sssq DataFrame with df_combined_responses
df_questionnaire_responses = pd.concat([df_combined_responses, df_sssq_extended], axis=1)

# 5. Print the head and information of the final df_questionnaire_responses DataFrame
print("--- df_questionnaire_responses head ---")
print(df_questionnaire_responses.head())

print("\n--- df_questionnaire_responses info ---")
df_questionnaire_responses.info()


In [ ]:
print(f"Listing files in {cleaned_data_dir}:")
!ls -F {cleaned_data_dir}


In [ ]:
import os

# Ensure the directory exists before navigating
os.makedirs(cleaned_data_dir, exist_ok=True)

print(f"Navigating to: {cleaned_data_dir}")
%cd {cleaned_data_dir}


In [ ]:
import os

# Define the directory to save cleaned data
cleaned_data_dir = f'{s2_data_path}/cleaned_data'

# Create the directory if it does not exist
os.makedirs(cleaned_data_dir, exist_ok=True)

print(f"Saving cleaned data to: {cleaned_data_dir}")

# Save df_event_timings
df_event_timings.to_csv(f'{cleaned_data_dir}/df_event_timings.csv', index=False)
print("Saved df_event_timings.csv")

# Save df_questionnaire_responses
df_questionnaire_responses.to_csv(f'{cleaned_data_dir}/df_questionnaire_responses.csv', index=False)
print("Saved df_questionnaire_responses.csv")

# Save individual chest sensor DataFrames
for sensor_name, df in chest_dataframes.items():
    df.to_csv(f'{cleaned_data_dir}/chest_{sensor_name.lower()}.csv', index=False)
    print(f"Saved chest_{sensor_name.lower()}.csv")

# Save individual wrist sensor DataFrames
for sensor_name, df in wrist_dataframes.items():
    df.to_csv(f'{cleaned_data_dir}/wrist_{sensor_name.lower()}.csv', index=False)
    print(f"Saved wrist_{sensor_name.lower()}.csv")

print("All cleaned DataFrames saved successfully!")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1. Search for your notebook in Drive
# Replace 'data_cleaning' with a keyword from your actual file name if different
!find /content/drive/MyDrive -name "*data_cleaning*.ipynb"

# 2. Once you see the path in the output, copy it here:
# Example: !cp "/content/drive/MyDrive/Colab Notebooks/data_cleaning.ipynb" /content/
# (Make sure to use quotes if there are spaces in the name)


In [ ]:

    import numpy as np
    import scipy.signal
    import pandas as pd

    def resample_sensor_data(df, target_freq, current_freq):
        if 'timestamp' not in df.columns:
            print("Timestamp column missing, cannot resample based on time. Assuming uniform sampling.")
            num_samples = len(df)
            time_duration = num_samples / current_freq
            new_num_samples = int(time_duration * target_freq)

            resampled_df = pd.DataFrame()
            for col in df.columns:
                if np.issubdtype(df[col].dtype, np.number): # Only resample numeric columns
                    resampled_data = scipy.signal.resample(df[col].values, new_num_samples)
                    resampled_df[col] = resampled_data
                else:
                    # For non-numeric, we can't directly resample, maybe forward fill or skip
                    # For now, let's just carry over if index matches, though index won't align
                    pass
            # Need to create a new time index for resampled_df
            new_time_index = np.linspace(0, time_duration, new_num_samples, endpoint=False)
            # If we had an original start time, we'd add it here.
            # resampled_df['timestamp'] = new_time_index + (original_start_time if available)
            # Since we don't have original timestamp, we create a relative one.
            resampled_df.insert(0, 'relative_time', new_time_index)

        else:
            time_seconds = (df['timestamp'] - df['timestamp'].iloc[0]) / np.timedelta64(1, 's')
            time_duration = time_seconds.iloc[-1]
            new_num_samples = int(time_duration * target_freq)
            new_time_index = np.linspace(0, time_duration, new_num_samples, endpoint=False)

            resampled_df = pd.DataFrame()
            resampled_df['timestamp_new'] = pd.to_timedelta(new_time_index, unit='s') + df['timestamp'].iloc[0]

            for col in df.columns:
                if col != 'timestamp' and np.issubdtype(df[col].dtype, np.number):
                    resampled_data = np.interp(new_time_index, time_seconds.values, df[col].values)
                    resampled_df[col] = resampled_data
            resampled_df = resampled_df.rename(columns={'timestamp_new': 'timestamp'})

        return resampled_df

    # Assuming chest_dataframes and wrist_dataframes are already loaded
    # and contain DataFrames for each sensor, with original frequencies known.

    # Example frequencies (replace with actual frequencies if known and different)
    chest_freq = 700
    wrist_freqs = {'ACC': 32, 'BVP': 64, 'EDA': 4, 'TEMP': 4} # Example freqs for wrist sensors

    target_freq = 64

    print("Resampling chest dataframes...")
    for sensor_name, df in chest_dataframes.items():
        print(f"Resampling {sensor_name} from {chest_freq}Hz to {target_freq}Hz")
        chest_dataframes[sensor_name] = resample_sensor_data(df.copy(), target_freq, chest_freq)
        print(f"New shape of {sensor_name}: {chest_dataframes[sensor_name].shape}")

    print("\nResampling wrist dataframes...")
    for sensor_name, df in wrist_dataframes.items():
        current_f = wrist_freqs.get(sensor_name, 32) # Default to 32 if not in map
        print(f"Resampling {sensor_name} from {current_f}Hz to {target_freq}Hz")
        wrist_dataframes[sensor_name] = resample_sensor_data(df.copy(), target_freq, current_f)
        print(f"New shape of {sensor_name}: {wrist_dataframes[sensor_name].shape}")

    print("\nResampling complete.")

